In [6]:
import pandas as pd

In [5]:
def NPV(oil_rate_list = [],well_start_list = [], l_well_list = []):
    if len(oil_rate_list) !=10 or len(well_start_list) !=10 or len(l_well_list) !=10:
        print("РќРµРІРµСЂРЅР°СЏ РґР»РёРЅР° СЃРїРёСЃРєР°")
        return None
    well_fix = 300
    L_well = 0.1
    Infr_fix = 5000
    oil_price = 0.01
    disc_rate  = 0.02
    tax = 0.5
    limit_oil = 5000000
    NPV_iter_LIST = []
    NPV = 0
    for i in range(10):
        rev = ((oil_rate_list[i] * oil_price )*(1-tax)/(1 +disc_rate )**i)  - max(0, oil_rate_list[i] - limit_oil )*oil_price
        if i == 0:
            expens = Infr_fix + well_start_list[i]*(well_fix + l_well_list[i]*L_well) 
        else:
            expens = (well_start_list[i]*(well_fix + l_well_list[i]*L_well))/(1 +disc_rate )**i 
        NPV_iter = rev - expens
        
        NPV+=NPV_iter
        NPV_iter_LIST.append(NPV_iter)
    return NPV

In [13]:
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from scipy.optimize import minimize

def parse_file(file_path):
    """
    Функция для парсинга текстового файла и извлечения данных.
    """
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    # Преобразование строк в числовые значения, игнорируя первую строку
    data = []
    for line in lines[1:]:
        values = list(map(float, line.strip().split()))
        data.extend(values)
    
    # Определение корректной формы массива
    data_size = len(data)
    expected_shape = (171, 121)
    
    if data_size != np.prod(expected_shape):
        raise ValueError(f"Размер данных ({data_size}) не соответствует ожидаемой форме {expected_shape}.")
    
    return np.array(data).reshape(expected_shape)

def load_property_maps(data_folder):
    """
    Загружает и парсит все файлы в папке, возвращает словарь с массивами numpy.
    """
    property_maps = {}
    
    for file_name in os.listdir(data_folder):
        if file_name.endswith('.txt'):
            file_path = os.path.join(data_folder, file_name)
            property_name = os.path.splitext(file_name)[0]
            try:
                property_maps[property_name] = parse_file(file_path)
            except ValueError as e:
                print(f"Ошибка при обработке файла {file_name}: {e}")
    
    return property_maps

def obtain_data(property_maps, well_coordinates, distances):
    """
    Функция для сбора геологических данных вдоль стволов скважин.
    """
    data = []
    for coord in well_coordinates:
        x, y = coord
        
        row = []
        for distance in distances:
            # Определяем границы вокруг точки с учетом дистанции
            x_start = max(x - distance, 0)
            x_end = min(x + distance, 171)
            y_start = max(y - distance, 0)
            y_end = min(y + distance, 121)
            
            for property_name, property_map in property_maps.items():
                region_values = property_map[x_start:x_end, y_start:y_end]
                mean_value = np.mean(region_values)
                row.append(mean_value)
        
        data.append(row)
    
    return pd.DataFrame(data, columns=[f"{prop}_{dist}" for prop in property_maps.keys() for dist in distances])

def predict_production(data_df, model):
    """
    Функция для предсказания добычи с использованием модели.
    """
    predictions = model.predict(data_df)
    return predictions

def calculate_npv(oil_rate_list, bhp_list, length_list):
    # Подключение функции расчета NPV (предполагается, что NPV_function определена)
    return NPV_function(oil_rate_list, bhp_list, length_list)

def optimize_wells(initial_wells, property_maps, model, max_iterations=100):
    def objective(well_params):
        well_coords = well_params[:len(initial_wells)*2].reshape(-1, 2)
        bhp = well_params[len(initial_wells)*2:len(initial_wells)*3]
        lengths = well_params[len(initial_wells)*3:]
        
        data_df = obtain_data(property_maps, well_coords, [1, 5, 10])
        oil_rate_predictions = model.predict(data_df)
        
        npv_value = calculate_npv(oil_rate_predictions, bhp, lengths)
        
        return -npv_value
    
    result = minimize(objective, initial_wells, method='L-BFGS-B', options={'maxiter': max_iterations})
    optimized_params = result.x
    
    return optimized_params, -result.fun

def generate_initial_wells():
    """
    Генерация начальных параметров для скважин (пример).
    """
    # Пример начальных координат и параметров скважин
    num_wells = 3  # Предположим, что у нас три скважины
    initial_coords = np.random.randint(0, 171, size=(num_wells, 2)).flatten()
    initial_bhp = np.random.uniform(100, 300, size=num_wells)
    initial_lengths = np.random.uniform(2000, 3000, size=num_wells)
    return np.concatenate([initial_coords, initial_bhp, initial_lengths])

def main_pipeline(data_folder, model):
    # Загрузка геологических карт
    property_maps = load_property_maps(data_folder)
    
    # Генерация начальных параметров для скважин
    initial_wells = generate_initial_wells()
    
    # Оптимизация расположения и параметров скважин
    optimized_wells, optimized_npv = optimize_wells(initial_wells, property_maps, model)
    
    return optimized_wells, optimized_npv

# Основной пайплайн
data_folder = "data"
model = CatBoostRegressor()
model.load_model("ML_model")

# Запуск основного пайплайна
optimized_wells, optimized_npv = main_pipeline(data_folder, model)
print("Optimized Wells:", optimized_wells)
print("Optimized NPV:", optimized_npv)


ValueError: cannot reshape array of size 20706 into shape (171,121)

In [20]:
#main_pipeline(data_folder, model)
model.feature_importances_#.predict([0,0,0,0,0,0,0,0])

array(None, dtype=object)